In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading, tempfile, os, sys, zipfile, requests, shutil
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
import numpy as np
from shapely.geometry import Point
import io
from IPython.display import Image as IPImage

# ─── State ────────────────────────────────────────────────────────────────
state = {
    'temp_dir':       tempfile.mkdtemp(),
    'zip_path':       None,
    'all_shapefiles': [],
    'shapefile_path': None,
    'predata_path':   None,
    'hbvpara_path':   None,
    'output_files':   {},
    'selected_id':    None,
    'id_col':         None,
    'gdf':            None,
}

# ─── Real-time log stream ─────────────────────────────────────────────────
class WidgetStream:
    def __init__(self, out): self.out = out
    def write(self, text): self.out.append_stdout(text)
    def flush(self): pass

# ─── Parse predata.csv ────────────────────────────────────────────────────
def parse_predata(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line == 'input_text':
                continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                rows.append((parts[0].strip(), parts[1].strip()))
    return dict(rows)

# ─── Step 1: Shapefile URL + Download ─────────────────────────────────────
shp_url  = widgets.Text(
    placeholder='https://wwwd3.ymparisto.fi/d3/gis_data/spesific/valumaalueet.zip',
    description='Shapefile URL:',
    layout=widgets.Layout(width='500px')
)
shp_btn  = widgets.Button(description='⬇ Download', button_style='info',
                           layout=widgets.Layout(width='120px'))
shp_prog = widgets.IntProgress(value=0, min=0, max=100,
                                layout=widgets.Layout(width='300px', visibility='hidden'))
shp_lbl  = widgets.Label(value='')

# ─── Step 2: Shapefile picker dropdown ────────────────────────────────────
shp_dropdown = widgets.Dropdown(
    description='Shapefile:',
    options=[],
    layout=widgets.Layout(width='500px', visibility='hidden')
)
shp_select_btn = widgets.Button(
    description='🗺 Load Shapefile',
    button_style='warning',
    layout=widgets.Layout(width='160px', visibility='hidden')
)
shp_pick_lbl = widgets.Label(value='')

# ─── Step 3: Map click area ───────────────────────────────────────────────
map_out      = widgets.Output(layout=widgets.Layout(
    border='1px solid #ddd', min_height='400px', visibility='hidden'
))
catchment_lbl = widgets.Label(value='')

# ─── Step 4: CSV uploads ──────────────────────────────────────────────────
upload_predata = widgets.FileUpload(accept='.csv', multiple=False,
                                     description='Upload',
                                     layout=widgets.Layout(width='200px'))
upload_hbvpara = widgets.FileUpload(accept='.csv', multiple=False,
                                     description='Upload',
                                     layout=widgets.Layout(width='200px'))
lbl_predata = widgets.Label(value='No file selected')
lbl_hbvpara = widgets.Label(value='No file selected')

def _save_csv(upload_w, state_key, label_w, change):
    val = change['new']
    if not val: return
    try:
        uploaded = val[0] if isinstance(val, tuple) else list(val.values())[0]
        fname    = uploaded.get('name', f'{state_key}.csv')
        content  = bytes(uploaded.get('content', b''))
        path     = os.path.join(state['temp_dir'], fname)
        with open(path, 'wb') as f:
            f.write(content)
        state[state_key] = path
        label_w.value = f'✅ {fname}'
    except Exception as e:
        label_w.value = f'❌ {e}'

upload_predata.observe(lambda c: _save_csv(upload_predata, 'predata_path', lbl_predata, c), names='value')
upload_hbvpara.observe(lambda c: _save_csv(upload_hbvpara, 'hbvpara_path', lbl_hbvpara, c), names='value')

# ─── Run / Log / Output ───────────────────────────────────────────────────
run_btn  = widgets.Button(description='▶ Run HBV Model', button_style='success',
                           layout=widgets.Layout(width='200px', height='40px'))
run_lbl  = widgets.Label(value='Complete all steps above then click Run.')
log_out  = widgets.Output(layout=widgets.Layout(
    border='1px solid #ccc', height='420px', overflow_y='scroll', padding='10px'
))
file_dd  = widgets.Dropdown(description='Output:', options=[], layout=widgets.Layout(width='420px'))
col_dd   = widgets.Dropdown(description='Column:', options=[], layout=widgets.Layout(width='420px'))
agg_tog  = widgets.ToggleButtons(options=['sum','mean'], description='Aggregate:', value='sum')
plot_btn = widgets.Button(description='📊 Plot on Map', button_style='info')
res_out  = widgets.Output()

# ─── Download handler ─────────────────────────────────────────────────────
def _do_download(b):
    url = shp_url.value.strip()
    if not url:
        shp_lbl.value = '❌ Enter a URL'
        return
    shp_btn.disabled = True
    shp_prog.layout.visibility = 'visible'
    shp_prog.value = 0
    shp_lbl.value  = '⏳ Connecting...'

    def _dl():
        try:
            fname = url.split('?')[0].split('/')[-1] or 'shapefile.zip'
            dest  = os.path.join(state['temp_dir'], fname)
            r = requests.get(url, stream=True, timeout=300)
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            done  = 0
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
                        done += len(chunk)
                        shp_prog.value = int(done/total*100) if total else 50
                        shp_lbl.value  = f'{done/1024/1024:.1f} MB'
            # Extract zip
            shp_lbl.value += ' — extracting...'
            exdir = os.path.join(state['temp_dir'], 'shp')
            os.makedirs(exdir, exist_ok=True)
            with zipfile.ZipFile(dest, 'r') as z:
                z.extractall(exdir)
            # Find all shapefiles
            shps = [os.path.join(dp, fn)
                    for dp, _, fns in os.walk(exdir)
                    for fn in fns if fn.endswith('.shp')]
            if not shps:
                shp_lbl.value = '❌ No .shp found in zip'
                return
            state['all_shapefiles'] = shps
            # Populate dropdown
            shp_dropdown.options = {os.path.basename(s): s for s in shps}
            shp_dropdown.layout.visibility   = 'visible'
            shp_select_btn.layout.visibility = 'visible'
            shp_prog.value = 100
            shp_lbl.value  = f'✅ Found {len(shps)} shapefiles — pick one below'
        except Exception as e:
            shp_lbl.value = f'❌ {e}'
        finally:
            shp_btn.disabled = False

    threading.Thread(target=_dl).start()

shp_btn.on_click(_do_download)

# ─── Load selected shapefile + draw clickable map ─────────────────────────
def _load_shapefile(b):
    path = shp_dropdown.value
    if not path:
        shp_pick_lbl.value = '❌ Select a shapefile first'
        return
    shp_select_btn.disabled = True
    shp_pick_lbl.value = '⏳ Loading...'

    try:
        gdf = gpd.read_file(path)
        # Find ID column
        id_col = next((c for c in gdf.columns if c.lower().endswith('_id')), None)
        if not id_col:
            shp_pick_lbl.value = '❌ No *_id column found in this shapefile'
            return
        state['shapefile_path'] = path
        state['gdf']     = gdf
        state['id_col']  = id_col
        shp_pick_lbl.value = f'✅ Loaded: {os.path.basename(path)} | ID column: {id_col} | {len(gdf)} catchments'
        _draw_map(gdf, id_col, selected_id=None)
        map_out.layout.visibility = 'visible'
        catchment_lbl.value = '👆 Click a catchment on the map to select it'
    except Exception as e:
        shp_pick_lbl.value = f'❌ {e}'
    finally:
        shp_select_btn.disabled = False

shp_select_btn.on_click(_load_shapefile)

# ─── Draw map ─────────────────────────────────────────────────────────────
def _draw_map(gdf, id_col, selected_id=None):
    map_out.clear_output(wait=True)
    with map_out:
        fig, ax = plt.subplots(figsize=(9, 7))
        gdf.plot(ax=ax, color='lightblue', edgecolor='black', linewidth=0.5)
        if selected_id is not None:
            gdf[gdf[id_col] == selected_id].plot(
                ax=ax, color='orange', edgecolor='black', linewidth=1.5
            )
        # Label each catchment with its ID
        for _, row in gdf.iterrows():
            centroid = row.geometry.centroid
            ax.annotate(
                str(row[id_col]),
                xy=(centroid.x, centroid.y),
                fontsize=6, ha='center', color='#333'
            )
        ax.set_title('Click a catchment to select it', fontsize=12)
        ax.set_axis_off()
        plt.tight_layout()

        # Convert to PNG and display as interactive image with click handler
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
        buf.seek(0)
        plt.close(fig)

        # Store figure bounds for click-to-coordinate mapping
        bounds = gdf.total_bounds  # (minx, miny, maxx, maxy)
        state['map_bounds'] = bounds
        state['map_buf']    = buf.getvalue()
        state['map_figsize'] = (9*120, 7*120)  # pixels at 120dpi

        display(IPImage(data=buf.getvalue()))
        print('\nOr type a catchment ID directly below:')

# ─── Manual catchment ID entry (fallback) ─────────────────────────────────
catchment_text = widgets.Text(
    placeholder='e.g. 1590307501.0',
    description='Catchment ID:',
    layout=widgets.Layout(width='320px')
)
catchment_confirm_btn = widgets.Button(
    description='✔ Confirm ID',
    button_style='warning',
    layout=widgets.Layout(width='120px')
)

def _confirm_id(b):
    gdf    = state.get('gdf')
    id_col = state.get('id_col')
    if gdf is None:
        catchment_lbl.value = '❌ Load a shapefile first'
        return
    raw = catchment_text.value.strip()
    # Try numeric match
    try:
        typed = float(raw)
        match = gdf[gdf[id_col] == typed]
        if match.empty:
            match = gdf[gdf[id_col].astype(str) == raw]
    except ValueError:
        match = gdf[gdf[id_col].astype(str) == raw]

    if match.empty:
        catchment_lbl.value = f'❌ ID "{raw}" not found. Valid IDs sample: {list(gdf[id_col].values[:5])}'
        return

    selected_id = match[id_col].values[0]
    state['selected_id'] = selected_id
    catchment_lbl.value  = f'✅ Selected catchment: {selected_id}'
    _draw_map(gdf, id_col, selected_id)

catchment_confirm_btn.on_click(_confirm_id)

# ─── Assemble tabs ────────────────────────────────────────────────────────
input_tab = widgets.VBox([
    widgets.HTML('<h3>📂 Step 1 — Catchment Shapefile</h3>'),
    widgets.HTML('<i>Paste the URL to the shapefile .zip and click Download</i>'),
    widgets.HBox([shp_url, shp_btn]),
    widgets.HBox([shp_prog, shp_lbl]),

    widgets.HTML('<h3>🗺 Step 2 — Pick a Shapefile</h3>'),
    widgets.HBox([shp_dropdown, shp_select_btn]),
    shp_pick_lbl,

    widgets.HTML('<h3>📍 Step 3 — Select Catchment</h3>'),
    map_out,
    catchment_lbl,
    widgets.HBox([catchment_text, catchment_confirm_btn]),

    widgets.HTML('<h3>📄 Step 4 — Upload CSV Files</h3>'),
    widgets.HTML('<b>predata.csv</b> — paths to NetCDF and land-use files'),
    widgets.HBox([upload_predata, lbl_predata]),
    widgets.HTML('<b>hbv_para.csv</b> — model parameters'),
    widgets.HBox([upload_hbvpara, lbl_hbvpara]),

    widgets.HTML('<hr>'),
    run_btn,
    run_lbl,
], layout=widgets.Layout(padding='16px'))

log_tab = widgets.VBox([
    widgets.HTML('<h3>📋 Model Logs</h3>'),
    log_out
], layout=widgets.Layout(padding='16px'))

output_tab = widgets.VBox([
    widgets.HTML('<h3>📈 Results</h3>'),
    file_dd, col_dd, agg_tog, plot_btn, res_out
], layout=widgets.Layout(padding='16px'))

tabs = widgets.Tab(children=[input_tab, log_tab, output_tab])
tabs.set_title(0, '📂 Input')
tabs.set_title(1, '📋 Logs')
tabs.set_title(2, '📈 Output')
display(tabs)

# ─── Run model ────────────────────────────────────────────────────────────
def run_model(b):
    if not state.get('selected_id'):
        run_lbl.value = '❌ Please select a catchment first (Step 3)'
        return
    run_btn.disabled = True
    run_lbl.value    = '⏳ Running...'
    log_out.clear_output()
    res_out.clear_output()
    tabs.selected_index = 1

    def _run():
        orig = sys.stdout
        sys.stdout = WidgetStream(log_out)
        try:
            # Validate
            if not state['shapefile_path']:
                print('❌ No shapefile loaded'); return
            if not state['predata_path']:
                print('❌ Please upload predata.csv'); return
            if not state['hbvpara_path']:
                print('❌ Please upload hbv_para.csv'); return

            print(f'✅ Shapefile:      {os.path.basename(state["shapefile_path"])}')
            print(f'✅ Catchment ID:   {state["selected_id"]}')
            print(f'✅ predata.csv:    {os.path.basename(state["predata_path"])}')
            print(f'✅ hbv_para.csv:   {os.path.basename(state["hbvpara_path"])}')

            # Parse predata.csv
            print('\n🔧 Parsing predata.csv...')
            cfg = parse_predata(state['predata_path'])
            print('   Keys:', list(cfg.keys()))

            # Fix relative paths
            for key, fname in [('output_csv_path','met_input.csv'),
                                ('csv_parameters_path','hbv_para.csv')]:
                if key in cfg and not os.path.isabs(cfg[key]):
                    cfg[key] = os.path.join(state['temp_dir'], fname)
                    print(f'   {key} → {cfg[key]}')

            # Copy hbv_para.csv if needed
            src = os.path.abspath(state['hbvpara_path'])
            dst = os.path.abspath(cfg['csv_parameters_path'])
            if src != dst:
                shutil.copy(src, dst)
                print(f'   Copied hbv_para.csv → {dst}')
            else:
                print(f'   hbv_para.csv already in place')

            # Run hbv_prepare
            print('\n⚙️  Running hbv_prepare...')
            from hbv_prepare import prepare_meteorological_and_landuse_data_direct
            output_csv = prepare_meteorological_and_landuse_data_direct(
                shapefile_path         = state['shapefile_path'],
                catchment_id_name      = state['id_col'],
                taso_id_of_interest    = state['selected_id'],
                precipitation_nc       = cfg['precipitation_nc_file_path'],
                evapotranspiration_nc  = cfg['evapotranspiration_nc_file_path'],
                temperature_nc         = cfg['temperature_nc_file_path'],
                output_csv_path        = cfg['output_csv_path'],
                urban_land_path        = cfg['urban_land_path'],
                agricultural_land_path = cfg['agricultural_land_path'],
                csv_parameters_path    = cfg['csv_parameters_path'],
            )
            print(f'✅ Met data: {output_csv}')

            # Run HBV
            print('\n🌊 Running HBV model...')
            os.chdir(os.path.dirname(cfg['csv_parameters_path']))
            from hbv_S2S import run_hbv_model
            output_files = run_hbv_model(output_csv)
            state['output_files'] = output_files
            print('\n✅ HBV model complete!')
            for k, v in output_files.items():
                print(f'   {k}: {v}')

            file_dd.options     = {k: v for k, v in output_files.items()}
            tabs.selected_index = 2
            run_lbl.value       = '✅ Done! Check the Output tab.'

        except Exception as e:
            import traceback
            print(f'\n❌ Error: {e}')
            print(traceback.format_exc())
            run_lbl.value = f'❌ {e}'
        finally:
            sys.stdout       = orig
            run_btn.disabled = False

    threading.Thread(target=_run).start()

run_btn.on_click(run_model)

# ─── Output tab ───────────────────────────────────────────────────────────
def on_file_change(change):
    if change['new']:
        df = pd.read_csv(change['new'])
        col_dd.options = df.select_dtypes(include='number').columns.tolist()

file_dd.observe(on_file_change, names='value')

def on_plot(b):
    res_out.clear_output()
    with res_out:
        try:
            df  = pd.read_csv(file_dd.value)
            col = col_dd.value
            agg = agg_tog.value
            val = df[col].mean() if agg == 'mean' else df[col].sum()
            print(f'{agg.title()} of "{col}": {val:.4f}')

            gdf = gpd.read_file(state['shapefile_path'])
            gdf['plot_value'] = 0.0
            gdf.loc[gdf[state['id_col']] == state['selected_id'], 'plot_value'] = val

            fig, ax = plt.subplots(figsize=(8, 6))
            gdf.plot(ax=ax, column='plot_value', cmap='Blues',
                     legend=True, edgecolor='black',
                     missing_kwds={'color': 'lightgrey'})
            ax.set_title(f'{agg.title()} of {col} — Catchment {state["selected_id"]}')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'❌ Plot error: {e}')

plot_btn.on_click(on_plot)